# Ansatz A - regelbasierte Transformation

**Concept:** Strict if-then rules and naming convention mapping. No LLM calls, no probabilistic decisions—all mappings are deterministic and traceable.

**Design decisions:**
- **Name-based mapping** with ID fallback (as decided in the discussion)

- **Mapping rules centralized in `COMPONENT_MAP`** — each component gets its own entry with props, value, and variant rules

- **Explicit skip properties** — all pure Figma visualization props (`State`, `Hover`, `Focus`, etc.) are filtered out

- **Sub-instances with an `_` prefix** are not rendered independently (they are children of components such as `password`/`inputtext`)

- **Wrapper frames (Column/Row/content)** are translated into `<div>` with Tailwind classes


In [18]:
import re
import json
from typing import Any
from pathlib import Path
from collections import Counter
from dataclasses import dataclass, field

In [19]:
INPUT_DIR = 'dataset/figma-data/cleaned/simple'
OUTPUT_DIR = 'dataset/storybook/src/stories/simple'

## 1. Components-Mapping-Configuration

In [20]:
UNIVERSAL_SKIP = {'State', 'Hover', 'Focus', 'Pressed', 'Active'}

COMPONENT_MAP: dict[str, dict] = {
    # === Button ===
    'button': {
        'primevue': 'Button',
        'props': {
            'Text#4293:477': {'target': 'label', 'type': 'text'},
            'Severity': {
                'target': 'severity', 'type': 'enum',
                'value_map': {
                    'Primary': None,  # Default — Omit prop
                    'Secondary': 'secondary', 'Success': 'success',
                    'Info': 'info', 'Warning': 'warn',
                    'Help': 'help', 'Danger': 'danger',
                    'Contrast': 'contrast',
                },
            },
            'Disabled': {'target': 'disabled', 'type': 'boolean', 'omit_when': False},
            'Icon Only': {'target': 'iconOnly', 'type': 'boolean', 'omit_when': False},
        },
        'variant_resolver': {
            'inputs': ['🔲 Outlined', '🔤 Text', 'Link', '⬆️ Raised', '⥰ Rounded'],
            'target': 'variant',
            'rules': [
                {'when': {'🔲 Outlined': 'True'}, 'result': 'outlined'},
                {'when': {'🔤 Text': 'True'}, 'result': 'text'},
                {'when': {'Link': 'True'}, 'result': 'link'},
            ],
        },
        'skip': UNIVERSAL_SKIP | {
            'Show Right Icon#1644:1387', 'Show Left Icon#1644:0',
            'Right Icon#1644:4161', 'Left Icon#1644:2774', 'Icon#1690:0',
        },
        'slot_strategy': 'drop',  # Children are Figma's built-in icon components
    },

    # === InputText ===
    'inputtext': {
        'primevue': 'InputText',
        'props': {
            '↳ Float Label#4275:152': {'target': 'placeholder', 'type': 'text'},
            '🚫 Disabled': {'target': 'disabled', 'type': 'boolean', 'omit_when': False},
            '❌ Invalid': {'target': 'invalid', 'type': 'boolean', 'omit_when': False},
            '🟦 Filled': {'target': 'variant', 'type': 'enum',
                         'value_map': {'False': None, 'True': 'filled'}},
            '🤏 Size': {'target': 'size', 'type': 'enum',
                       'value_map': {'Normal': None, 'Small': 'small', 'Large': 'large'}},
        },
        'skip': UNIVERSAL_SKIP | {
            '⚙️ State', 'ℹ️ Show Helper#1729:1731', '↳ Helper Text#1729:4039',
            '↳ Label#5662:83', '🏷️ Show Label#5662:50',
            '➡️ Show Right Icon#1729:3462', '↳ Right Icon#1729:577',
            '⬅️ Show Left Icon#1729:2885', '↳ Left Icon#1729:0',
            '👁️ Show Text#1729:2308', '▭ Ifta Label', '🏷️ Float Label',
            '↳ Float Label Variant',
        },
        'slot_strategy': 'drop',  # Internal structure is rendered by PrimeVue
        'extra_attrs': {'v-model': '_state.{node_id}'},  # 2-way binding placeholder
    },

    # === Password ===
    'password': {
        'primevue': 'Password',
        'props': {
            'Toggle Mask': {'target': 'toggleMask', 'type': 'boolean', 'omit_when': False},
        },
        'skip': UNIVERSAL_SKIP | {'Password Visible'},
        'slot_strategy': 'drop',
        'extra_attrs': {'v-model': '_state.{node_id}'},
        # Placeholder  comes from the internal _inputtext-content sub-instance
        'inherit_placeholder_from_sub': '_inputtext-content',
    },

    # === InputNumber ===
    'inputnumber': {
        'primevue': 'InputNumber',
        'props': {
            'Type': {'target': 'buttonLayout', 'type': 'enum',
                     'value_map': {'Horizontal': 'horizontal', 'Vertical': 'vertical',
                                   'Stacked': 'stacked', 'Default': None}},
        },
        'skip': UNIVERSAL_SKIP,
        'slot_strategy': 'drop',
        'extra_attrs': {'v-model': '_state.{node_id}', 'showButtons': True},
        'inherit_placeholder_from_sub': '_inputtext-content',
    },

    # === Avatar ===
    'avatar': {
        'primevue': 'Avatar',
        'props': {
            'Text#4271:0': {'target': 'label', 'type': 'text'},
            'Size': {'target': 'size', 'type': 'enum',
                     'value_map': {'Normal': None, 'Large': 'large', 'X-Large': 'xlarge'}},
            'Circle': {'target': 'shape', 'type': 'enum',
                       'value_map': {'True': 'circle', 'False': None}},
            'Type': {'target': '_type', 'type': 'enum',
                     'value_map': {'Label': None, 'Icon': None, 'Image': None}},
        },
        'skip': UNIVERSAL_SKIP,
        'slot_strategy': 'drop',  # Text is set via the label property; badge is set separately (see below)
        'overlay_badge_child': True,  # If Show Badge=True → wrap with OverlayBadge
    },

    # === OverlayBadge ===
    'overlaybadge': {
        'primevue': 'OverlayBadge',
        'props': {
            'Text#4272:0': {'target': 'value', 'type': 'text'},
            'Severity': {
                'target': 'severity', 'type': 'enum',
                'value_map': {'Primary': None, 'Secondary': 'secondary',
                              'Success': 'success', 'Info': 'info',
                              'Warning': 'warn', 'Danger': 'danger',
                              'Contrast': 'contrast'},
            },
            'Size': {'target': 'size', 'type': 'enum',
                     'value_map': {'Default': None, 'Large': 'large', 'X-Large': 'xlarge'}},
        },
        'skip': UNIVERSAL_SKIP | {'Circle'},
        'slot_strategy': 'drop',
    },

    # === Slider ===
    'slider': {
        'primevue': 'Slider',
        'props': {
            'Direction': {'target': 'orientation', 'type': 'enum',
                          'value_map': {'Horizontal': None, 'Vertical': 'vertical'}},
            'Disabled': {'target': 'disabled', 'type': 'boolean', 'omit_when': False},
            'Range': {'target': 'range', 'type': 'boolean', 'omit_when': False},
        },
        'skip': UNIVERSAL_SKIP | {'Input', 'Slide'},
        'slot_strategy': 'drop',
        'extra_attrs': {'v-model': '_state.{node_id}'},
    },
}

print(f'Components in Mapping: {len(COMPONENT_MAP)}')
for name in sorted(COMPONENT_MAP.keys()):
    pv = COMPONENT_MAP[name]["primevue"]
    print(f'  {name:25s} -> <{pv}>')

Components in Mapping: 7
  avatar                    -> <Avatar>
  button                    -> <Button>
  inputnumber               -> <InputNumber>
  inputtext                 -> <InputText>
  overlaybadge              -> <OverlayBadge>
  password                  -> <Password>
  slider                    -> <Slider>


## 2. AST-Definition

In [21]:
@dataclass
class UINode:
    tag: str  # 'Button', 'div', etc.
    props: dict = field(default_factory=dict)  # static Props
    dynamic_props: dict = field(default_factory=dict)  # with `:`-Prefix
    classes: list[str] = field(default_factory=list)  # Tailwind-Classes
    children: list = field(default_factory=list)  # more UINodes or Strings
    slot: str | None = None  # if a child of a component slot
    is_component: bool = False  # PrimeVue-Component vs. HTML-Element
    self_closing: bool = False
    figma_id: str | None = None  # Tracking for Debugging
    figma_name: str | None = None

    def __repr__(self):
        return f'UINode({self.tag}, props={list(self.props)}, children={len(self.children)})'

## 3. Helper: Apply Property-Mapping

In [22]:
def _extract_value(prop_entry: Any) -> Any:
    if isinstance(prop_entry, dict) and 'value' in prop_entry:
        return prop_entry['value']

    return prop_entry


def _convert_value(raw: Any, prop_type: str) -> Any:
    if prop_type == 'boolean':
        if isinstance(raw, bool):
            return raw
        if isinstance(raw, str):
            return raw.lower() == 'true'

    if prop_type == 'number':
        try:
            return float(raw)
        except (TypeError, ValueError):
            return None

    return raw

In [23]:
def apply_property_rules(figma_props: dict, spec: dict) -> tuple[dict, dict]:
    """Mapping component through mapping rules

    Returns:
        (static_props, dynamic_props) — separated by string- and boolean-values
    """
    static = {}
    dynamic = {}
    skip = spec.get('skip', set())
    rules = spec.get('props', {})

    # 1) Simple 1:1-Mappings
    for figma_key, rule in rules.items():
        if figma_key not in figma_props:
            continue
        raw = _extract_value(figma_props[figma_key])
        prop_type = rule.get('type', 'text')
        target = rule['target']

        # Value-Mapping (example: 'Primary' -> None)
        if 'value_map' in rule:
            if raw not in rule['value_map']:
                continue
            mapped = rule['value_map'][raw]
            if mapped is None:
                continue  # Default-Value → omit
            value = mapped
        else:
            value = _convert_value(raw, prop_type)

        if value is None:
            continue
        if 'omit_when' in rule and value == rule['omit_when']:
            continue

        # Booleans / Numbers over dynamic-binding return
        if prop_type == 'boolean':
            dynamic[target] = 'true' if value else 'false'
        elif prop_type == 'number':
            dynamic[target] = str(value)
        else:
            static[target] = str(value)

    # 2) Variant-Resolver: several Booleans -> one Enum-Prop
    vr = spec.get('variant_resolver')
    if vr:
        for rule in vr['rules']:
            match = all(_extract_value(figma_props.get(k)) == v
                        for k, v in rule['when'].items())
            if match:
                static[vr['target']] = rule['result']
                break

    return static, dynamic

## 4. Layout-Engine: FRAME → Tailwind

In [24]:
TAILWIND_SPACING = {
    0: '0', 1: '0.5', 2: '0.5', 4: '1', 6: '1.5', 8: '2', 10: '2.5', 12: '3',
    14: '3.5', 16: '4', 20: '5', 24: '6', 28: '7', 32: '8', 36: '9', 40: '10',
    44: '11', 48: '12', 56: '14', 64: '16',
}


def _spacing_class(prefix: str, px: float | int | None) -> str | None:
    """Provides for example 'gap-4' for 16px, 'p-6' for 24px"""
    if px is None or px == 0:
        return None

    px_int = int(round(px))
    if px_int in TAILWIND_SPACING:
        return f'{prefix}-{TAILWIND_SPACING[px_int]}'

    # Fallback: arbitrary value
    return f'{prefix}-[{px_int}px]'

In [25]:
def frame_to_classes(node: dict) -> list[str]:
    """Converts FRAME-layout-properties to Tailwind classes"""
    classes = []
    layout_mode = node.get('layoutMode')

    if layout_mode == 'HORIZONTAL':
        classes.append('flex')
    elif layout_mode == 'VERTICAL':
        classes.append('flex flex-col')

    # Spacing
    gap = _spacing_class('gap', node.get('itemSpacing'))
    if gap:
        classes.append(gap)

    # Padding — Simplify symmetrically if possible
    pl, pr = node.get('paddingLeft'), node.get('paddingRight')
    pt, pb = node.get('paddingTop'), node.get('paddingBottom')
    if pl == pr == pt == pb and pl:
        c = _spacing_class('p', pl)

        if c: classes.append(c)
    else:
        if pl == pr and pl:
            c = _spacing_class('px', pl)

            if c: classes.append(c)
        else:
            for px, prefix in [(pl, 'pl'), (pr, 'pr')]:
                c = _spacing_class(prefix, px)

                if c: classes.append(c)
        if pt == pb and pt:
            c = _spacing_class('py', pt)

            if c: classes.append(c)
        else:
            for px, prefix in [(pt, 'pt'), (pb, 'pb')]:
                c = _spacing_class(prefix, px)

                if c: classes.append(c)

    # Cross-Axis Alignment
    counter = node.get('counterAxisAlignItems')
    if counter == 'CENTER':
        classes.append('items-center')
    elif counter == 'MAX':
        classes.append('items-end')

    # Primary-Axis Alignment
    primary = node.get('primaryAxisAlignItems')
    if primary == 'CENTER':
        classes.append('justify-center')
    elif primary == 'MAX':
        classes.append('justify-end')
    elif primary == 'SPACE_BETWEEN':
        classes.append('justify-between')

    return classes

## 5. AST-Builder: Figma-JSON → UINode


In [26]:
def _normalize_name(name: str) -> str:
    """'Input Text' / 'input-text' / 'InputText' -> 'inputtext'"""
    return re.sub(r'[\s\-_]+', '', name or '').lower()


def _find_child_by_name(figma_node: dict, target_name: str) -> dict | None:
    for c in figma_node.get('children', []) or []:
        if _normalize_name(c.get('name', '')) == _normalize_name(target_name):
            return c

    return None


def _find_descendant_by_name(figma_node: dict, target_name: str) -> dict | None:
    """Recursive search throughout the entire subtree"""
    target = _normalize_name(target_name)

    for c in figma_node.get('children', []) or []:
        if _normalize_name(c.get('name', '')) == target:
            return c

        found = _find_descendant_by_name(c, target_name)

        if found is not None:
            return found

    return None

In [27]:
def _transform_instance(node: dict) -> UINode | None:
    """A Figma instance is mapped to a PrimeVue component"""
    name_key = _normalize_name(node.get('name', ''))
    spec = COMPONENT_MAP.get(name_key)

    if not spec:
        # Unknown Component — Display as a generic div with a comment
        return UINode(
            tag='div',
            classes=['border', 'border-dashed', 'border-red-400', 'p-2'],
            children=[f'<!-- Unmapped INSTANCE: {node.get("name")!r} -->'],
            figma_id=node.get('id'),
            figma_name=node.get('name'),
        )

    figma_props = node.get('componentProperties', {}) or {}
    static, dynamic = apply_property_rules(figma_props, spec)

    # extra_attrs (e.g. example v-model)
    extra = spec.get('extra_attrs', {})
    nid = 'n' + (node.get('id') or '').replace(':', '_').replace(';', '_').replace('-', '_')
    for k, v in extra.items():
        if isinstance(v, str) and '{node_id}' in v:
            v = v.replace('{node_id}', nid)

        if isinstance(v, bool):
            if v:
                static[k] = ''  # # Boolean attribute without a value
        else:
            static[k] = v

    # Inherit placeholder from a sub-instance (e.g. password.placeholder aus _inputtext-content)
    sub_name = spec.get('inherit_placeholder_from_sub')
    if sub_name and 'placeholder' not in static:
        sub = _find_descendant_by_name(node, sub_name)

        if sub:
            sub_props = sub.get('componentProperties', {}) or {}
            text_config = _extract_value(sub_props.get('Text Config'))

            if text_config == 'Placeholder':
                ph = _extract_value(sub_props.get('Placeholder#4275:140'))

                if ph:
                    static['placeholder'] = str(ph)
            else:
                val = _extract_value(sub_props.get('Value#4275:146'))

                if val:
                    static['value'] = str(val)

    # Slot-Strategy
    children = []
    if spec.get('slot_strategy') == 'default':
        for c in node.get('children', []) or []:
            sub = transform_node(c)

            if sub:
                children.append(sub)

    # Avatar with OverlayBadge
    if spec.get('overlay_badge_child'):
        show_badge = _extract_value(figma_props.get('Show Badge#2138:0', {})) == True

        if show_badge:
            badge_node = _find_child_by_name(node, 'overlaybadge')

            if badge_node:
                badge_ui = transform_node(badge_node)

                if badge_ui:
                    # Avatar becomes the default slot for OverlayBadge
                    avatar_ui = UINode(
                        tag=spec['primevue'], props=static, dynamic_props=dynamic,
                        self_closing=True, is_component=True,
                        figma_id=node.get('id'), figma_name=node.get('name'),
                    )

                    badge_ui.children = [avatar_ui]
                    badge_ui.self_closing = False

                    return badge_ui

    return UINode(
        tag=spec['primevue'],
        props=static,
        dynamic_props=dynamic,
        children=children,
        is_component=True,
        self_closing=not children,
        figma_id=node.get('id'),
        figma_name=node.get('name'),
    )

In [28]:
def _transform_frame(node: dict) -> UINode | None:
    """Wrapper-FRAME → <div> with Tailwind-Classes"""
    classes = frame_to_classes(node)
    children = []

    for c in node.get('children', []) or []:
        sub = transform_node(c)

        if sub:
            children.append(sub)

    # If there are no classes and only one child → Pass the frame through transparently
    if not classes and len(children) == 1:
        return children[0]

    return UINode(
        tag='div',
        classes=classes,
        children=children,
        figma_id=node.get('id'),
        figma_name=node.get('name'),
    )

In [29]:
def _transform_text(node: dict) -> UINode | None:
    """TEXT node — if the text is NOT property-bound, render it as <span>"""
    if node.get('componentPropertyReferences', {}).get('characters'):
        # Text is set by the parent component via a prop → no separate node
        return None

    text = node.get('characters', '')

    if not text:
        return None

    # Font size mapping (simplified)
    style = node.get('style', {})
    classes = []
    fs = style.get('fontSize')

    if fs:
        size_map = {12: 'text-xs', 14: 'text-sm', 16: 'text-base',
                    18: 'text-lg', 20: 'text-xl', 24: 'text-2xl', 28: 'text-3xl'}

        cls = size_map.get(int(fs))

        if cls:
            classes.append(cls)

    fw = style.get('fontWeight', 400)

    if fw and int(fw) >= 600:
        classes.append('font-semibold' if int(fw) < 700 else 'font-bold')

    return UINode(
        tag='span',
        classes=classes,
        children=[text],
        figma_id=node.get('id'),
        figma_name=node.get('name')
    )

In [30]:
def transform_node(figma_node: dict) -> UINode | None:
    """Converts a single Figma node into a UINode

    Returns None if the node is to be discarded (e.g., internal sub-instances)
    """
    if not isinstance(figma_node, dict):
        return None

    node_type = figma_node.get('type')
    raw_name = figma_node.get('name', '')

    # Internal Helper-Instances (Prefix '_') are dropped
    if raw_name.startswith('_'):
        return None

    if node_type == 'INSTANCE':
        return _transform_instance(figma_node)
    elif node_type == 'FRAME':
        return _transform_frame(figma_node)
    elif node_type == 'TEXT':
        return _transform_text(figma_node)
    else:
        # VECTOR, RECTANGLE, etc. – no treatment at this time
        return None

## 6. Code-Generator: UINode → Vue 3 SFC

In [31]:
def render_ast(node: UINode, depth: int = 0, imports: set | None = None) -> str:
    if imports is None:
        imports = set()

    indent = '  ' * depth

    if isinstance(node, str):
        return f'{indent}{node}'

    if node.is_component:
        imports.add(node.tag)

    # Assembling attributes
    attrs = []

    if node.classes:
        attrs.append(f'class="{" ".join(node.classes)}"')

    for k, v in node.props.items():
        if v == '':
            attrs.append(k)  # Boolean attribute without a value
        else:
            attrs.append(f'{k}="{v}"')

    for k, v in node.dynamic_props.items():
        attrs.append(f':{k}="{v}"')

    attr_str = ''

    if attrs:
        # If there are more than 2 attributes: Multi-line
        if len(attrs) > 2 or sum(len(a) for a in attrs) > 60:
            attr_str = '\n' + '\n'.join(f'{indent}  {a}' for a in attrs) + f'\n{indent}'
        else:
            attr_str = ' ' + ' '.join(attrs)

    if node.self_closing or not node.children:
        return f'{indent}<{node.tag}{attr_str} />'

    # With children
    opening = f'{indent}<{node.tag}{attr_str}>'
    closing = f'{indent}</{node.tag}>'

    rendered_children = []

    for c in node.children:
        if isinstance(c, str):
            # String-Child inline
            rendered_children.append(f'{indent}  {c}')
        else:
            rendered_children.append(render_ast(c, depth + 1, imports))

    # Single string child → inline
    if len(node.children) == 1 and isinstance(node.children[0], str):
        return f'{indent}<{node.tag}{attr_str}>{node.children[0]}</{node.tag}>'

    return opening + '\n' + '\n'.join(rendered_children) + '\n' + closing


def generate_sfc(figma_root: dict) -> str:
    """Generates a complete Vue 3 SFC from a Figma mockup"""
    ast = transform_node(figma_root)

    if ast is None:
        return '<!-- Unable to transform the mockup -->'

    imports: set[str] = set()
    template_body = render_ast(ast, depth=1, imports=imports)

    # Collect state bindings (for v-model placeholders)
    state_refs = []

    def collect_refs(n):
        if isinstance(n, str): return

        for k, v in n.props.items():
            if k == 'v-model' and isinstance(v, str) and v.startswith('_state.'):
                state_refs.append(v.split('.', 1)[1])

        for c in n.children:
            collect_refs(c)

    collect_refs(ast)

    # Script-Setup
    script_lines = ['<script setup>']
    if state_refs:
        script_lines.append("import { reactive } from 'vue'")

    for imp in sorted(imports):
        script_lines.append(f"import {imp} from 'primevue/{imp.lower()}'")

    if state_refs:
        script_lines.append('')
        init = ', '.join(f'{r}: null' for r in state_refs)
        script_lines.append(f'const _state = reactive({{ {init} }})')

    script_lines.append('</script>')

    template = f'<template>\n{template_body}\n</template>'

    return template + '\n\n' + '\n'.join(script_lines) + '\n'

## 7. Apply the pipeline to all mockups

In [32]:
INPUT_DIR_PATH = Path(INPUT_DIR)
OUTPUT_DIR_PATH = Path(OUTPUT_DIR)

input_files = sorted(INPUT_DIR_PATH.rglob('*.json'))
print(f'Input-Files: {len(input_files)}\n')

results = []

for path in input_files:
    with open(path, 'r', encoding='utf-8') as f:
        figma = json.load(f)

    sfc = generate_sfc(figma)

    rel = path.relative_to(INPUT_DIR).with_stem(path.stem + '-a').with_suffix('.vue')
    out_path = OUTPUT_DIR_PATH / rel
    out_path.parent.mkdir(parents=True, exist_ok=True)

    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(sfc)

    results.append((path, out_path, len(sfc)))

    print(f'  {path.name:40s} -> {out_path.name:40s}  ({len(sfc):4d} bytes)')

print(f'\nGenerated: {len(results)} SFCs')

Input-Files: 10

  1.json                                   -> 1-a.vue                                   ( 456 bytes)
  10.json                                  -> 10-a.vue                                  (1343 bytes)
  2.json                                   -> 2-a.vue                                   ( 429 bytes)
  3.json                                   -> 3-a.vue                                   ( 497 bytes)
  4.json                                   -> 4-a.vue                                   ( 878 bytes)
  5.json                                   -> 5-a.vue                                   ( 420 bytes)
  6.json                                   -> 6-a.vue                                   ( 347 bytes)
  7.json                                   -> 7-a.vue                                   ( 557 bytes)
  8.json                                   -> 8-a.vue                                   ( 479 bytes)
  9.json                                   -> 9-a.vue                     

## 8. Random Inspection

In [33]:
SAMPLE = '01.vue'
sample_path = next((op for _, op, _ in results if op.name == SAMPLE), None)

if sample_path is None and results:
    sample_path = results[0][1]

print(f'=== {sample_path.name} ===\n')
print(sample_path.read_text(encoding='utf-8'))

=== 1-a.vue ===

<template>
  <div class="flex flex-col p-6">
    <div class="flex flex-col gap-6">
      <Password
        v-model="_state.n10_4744"
        placeholder="Passwort eingeben"
        :toggleMask="true"
       />
      <Button label="Anmelden" />
    </div>
  </div>
</template>

<script setup>
import { reactive } from 'vue'
import Button from 'primevue/button'
import Password from 'primevue/password'

const _state = reactive({ n10_4744: null })
</script>



## 10. Coverage-Diagnose

In [34]:
mapped = Counter()
unmapped = Counter()

def diagnose(figma_node, inside_dropped: bool = False):
    if not isinstance(figma_node, dict):
        return

    if figma_node.get('type') == 'INSTANCE':
        name = figma_node.get('name', '')

        if name.startswith('_'):
            return  # Internal sub-instance, intentionally dropped

        if inside_dropped:
            return  # Children of a drop-strategy component are intentionally ignored

        key = _normalize_name(name)

        if key in COMPONENT_MAP:
            mapped[name] += 1
            spec = COMPONENT_MAP[key]

            # If slot_strategy=‘drop’: do not descend into Children
            if spec.get('slot_strategy') == 'drop' and not spec.get('overlay_badge_child'):
                return
        else:
            unmapped[name] += 1

    for c in figma_node.get('children', []) or []:
        diagnose(c, inside_dropped)

for path in input_files:
    with open(path, 'r', encoding='utf-8') as f:
        diagnose(json.load(f))

print(f'Mapped instances:   {sum(mapped.values()):4d}')
for n, c in mapped.most_common():
    print(f'  {n:25s} {c:3d}x')

print(f'\nUnmapped Instances: {sum(unmapped.values()):4d}')
for n, c in unmapped.most_common():
    print(f'  {n:25s} {c:3d}x  <- Add mapping to COMPONENT_MAP')

total = sum(mapped.values()) + sum(unmapped.values())
if total:
    print(f'\nCoverage: {sum(mapped.values()) / total * 100:.1f}%')

Mapped instances:     12
  button                      4x
  inputtext                   2x
  inputnumber                 2x
  password                    1x
  avatar                      1x
  overlaybadge                1x
  slider                      1x

Unmapped Instances:   27
  radiobutton                 7x  <- Add mapping to COMPONENT_MAP
  divider                     5x  <- Add mapping to COMPONENT_MAP
  checkbox                    4x  <- Add mapping to COMPONENT_MAP
  tag                         4x  <- Add mapping to COMPONENT_MAP
  skeleton                    4x  <- Add mapping to COMPONENT_MAP
  toggleswitch                1x  <- Add mapping to COMPONENT_MAP
  textarea                    1x  <- Add mapping to COMPONENT_MAP
  progressbar                 1x  <- Add mapping to COMPONENT_MAP

Coverage: 30.8%
